# Lab 04: GPU Accelerate Training

<p align="center">
  <img src="MONAI_LISA_5.png" width="400">
</p>

**MONAI_LISA** (Learning Image Synthesis & Analysis) is a personal medical AI sandbox dedicated to learning and mastering the [MONAI](https://project-monai.github.io/) framework. Built on top of MONAI, this project serves as a practical learning platform and playground for implementing, experimenting with, and benchmarking state-of-the-art 3D medical deep learning techniques. This Lab04 notebook is based on the **"fast_training_tutorial.ipynb"** tutorial from the **[Tutorial of Project-MONAI](https://github.com/Project-MONAI/tutorials/tree/main)**.

This practice module is divided into three parts:
1. Core Notebook (Lab04_GPU_Accelerate_Training.ipynb): A comprehensive Jupyter Notebook containing detailed explanations and the complete workflow for this exercise.

2. Profiling Notebook (Lab04_GPU_Accelerate_Training_colab.ipynb): A Jupyter Notebook designed to run on Google Colab or a local server for performance profiling. This part includes the complete environment setup, including NVIDIA Nsys configuration.

3. Profiling Script (Lab04_GPU_Accelerate_Training_colab.py): A streamlined Python script dedicated strictly to profiling analysis. It omits the environment setup and utilizes a reduced number of training epochs for quick execution. This script is intended to be run after completing the setup in the profiling notebook.


### 🔗 Useful Resources
*   **GitHub Repository:** [Tutorial of Project-MONAI](https://github.com/Project-MONAI/tutorials/tree/main)
*   **Video Tutorial:** [MONAI Bootcamp 2021 Playlist](https://www.youtube.com/watch?v=SWtBkxwDq94&list=PLtoSVSQ2XzyCobzE6NvwjNpITsQyPUtfs&index=6)

---
**Adapted by:** Yun-An Huang  
**Date:** June 8th, 2026


In [ ]:
## Initialization: Install MONAI and test the environment

from google.colab import drive # Import Google Colab drive utility
import os # Import os module for interacting with the operating system

# Install PyTorch with CUDA support and MONAI with all its optional dependencies
!pip install --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!pip install "monai[all]" nibabel pydicom ipywidgets==8.1.2


# Clone MONAI tutorials and Bootcamp repositories if they don't already exist
if not os.path.exists("tutorials"):
    !git clone https://github.com/Project-MONAI/tutorials.git
    !git clone https://github.com/Project-MONAI/MONAIBootcamp2021.git

# Ensure MONAI is installed and configured correctly
import monai

# Test environment
import torch

print("GPU Avalable:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU type:", torch.cuda.get_device_name(0))



In [ ]:
# avoid the confliction between numpy scipy scikit-learn

!pip install --upgrade numpy scipy scikit-learn

In [ ]:
# install nsys
!apt-get update && apt-get install -y nsight-systems-2026.1.3

In [ ]:
# Optional:
# if you are using google colab, the server would clean all the data after you disconnect.
# So, you may like to save the data in your personal space.
# One of the solution is mount Google Drive to persist trained model weights and logs

#from google.colab import drive # Import Google Colab drive utility
drive.mount('/content/drive/') # Mount Google Drive to access persistent storage

PROJECT_SANDBOX = "/content/drive/MyDrive/Colab_Workspace/MONAI_LISA" # Define the project sandbox directory within Google Drive

print('Current Dir:')
print(os.listdir(PROJECT_SANDBOX)) # List contents of the project sandbox directory to verify

In [ ]:
# Examine Nvidia Environment

!nvidia-smi

## Setup imports

In [ ]:
import glob
import math
import os
import shutil
import tempfile
import time

import matplotlib.pyplot as plt
import torch

!python -c "import monai" || pip install -q "monai-weekly[nibabel, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
%matplotlib inline


from torch.optim import Adam, SGD
from monai.apps import download_and_extract
from monai.config import print_config
from monai.data import (
    CacheDataset,
    DataLoader,
    ThreadDataLoader,
    Dataset,
    decollate_batch,
    set_track_meta,
)
from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss, DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.layers import Act, Norm
from monai.networks.nets import UNet
from monai.transforms import (
    EnsureChannelFirstd,
    AsDiscrete,
    Compose,
    CropForegroundd,
    EnsureTyped,
    FgBgToIndicesd,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    ScaleIntensityRanged,
    Spacingd,
)
from monai.utils import set_determinism

# for profiling
import nvtx
from monai.utils.nvtx import Range
import contextlib  # to improve code readability (combining training/validation loop with and without profiling)

print_config()

## Setup data & output directories

You can specify a directory with the `MONAI_DATA_DIRECTORY` environment variable.  
This allows you to save results and reuse downloads.  
If not specified a temporary directory will be used.

In [ ]:
directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory
print(f"root dir is: {root_dir}")

In [ ]:
# outputs

out_dir = os.path.join(PROJECT_SANDBOX,"Lab04","outputs/")

if not os.path.exists(out_dir):
    os.makedirs(out_dir)

print(out_dir)

nsys_output = os.path.join(out_dir, "output_base")

## download dataset

In [ ]:
resource = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar"
md5 = "410d4a301da4e5b2f6f86ec3ddba524e"

compressed_file = os.path.join(root_dir, "Task09_Spleen.tar")
data_root = os.path.join(root_dir, "Task09_Spleen")
if not os.path.exists(data_root):
    download_and_extract(resource, compressed_file, root_dir, md5)

## Profiling the script

In [ ]:
%%bash -s "$nsys_output"
#nsys profile --output "$1" --force-overwrite true --trace-fork-before-exec true python3 -u Lab04_GPU_Accelerate_Training_colab.py
nohup nsys profile --output "$1" --force-overwrite true --trace-fork-before-exec true python3 Lab04_GPU_Accelerate_Training_colab.py > nsys_log.txt 2>&1 &

In [ ]:
import time
from IPython.display import clear_output

# read the output of log for 120 times x 10s
for _ in range(120):
    clear_output(wait=True)
    print("====== Nsys System Log ======")
    !cat nsys_log.txt | tail -n 20
    time.sleep(10)